# 🎓 Train PhoBERT on ViFactCheck (Vietnamese Fake News)

**Notebook này thực hiện:**
1. Cài đặt môi trường & dependencies cho tiếng Việt (`underthesea`, `transformers`, v.v.)
2. Tải dataset **ViFactCheck** (AAAI 2025) từ HuggingFace.
3. Tiền xử lý văn bản tiếng Việt (Unicode, Word Segmentation).
4. Trích xuất **Stylistic Features** (word count, sentiment, exclamation ratio, v.v.).
5. Huấn luyện & So sánh 2 phiên bản model:
   - **Baseline**: PhoBERT-base (chỉ text).
   - **Pro (Improvements)**: PhoBERT + Concatenated Stylistic Features.

In [1]:
# 🔧 1. Cài đặt Dependencies
!pip install -q transformers datasets underthesea pyvi scikit-learn pandas tqdm torch
print("✅ Dependencies installed!")

# Kiểm tra GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 66.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.4 MB/s eta 0:00:00
✅ Dependencies installed!
CUDA available: False


In [ ]:
# 📥 2. Clone Repo & Config
import os
REPO_URL = "https://github.com/phidinhmanh/fake-news-detection.git" # Thay bằng repo của bạn
REPO_DIR = "fake-news-detection"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL

%cd $REPO_DIR
import sys
sys.path.append(os.getcwd())

In [ ]:
# 📥 3. Download ViFactCheck Dataset
# Chạy script download để lấy data từ HuggingFace
!python dataset/download_datasets.py

print("\n📊 Kiểm tra file vifactcheck_full.csv:")
!ls -lh data/raw/vfc/

In [ ]:
# 🔧 4. Preprocessing & Word Segmentation
# Sử dụng underthesea để tách từ tiếng Việt
!python dataset/preprocess_vietnamese.py

print("\n📊 Kiểm tra file preprocessed_all.csv:")
!ls -lh data/datasets/processed/

In [ ]:
# 🔗 5. Merge & Split (80/10/10)
# CHÚ Ý: Dùng flag --exclude-suspicious nếu muốn bỏ nhãn NEI (Not Enough Info)
EXCLUDE_NEI = True # @param {type:"boolean"}

cmd = "python dataset/merge_datasets.py"
if EXCLUDE_NEI:
    cmd += " --exclude-suspicious"

!$cmd

In [ ]:
# 📊 6. Feature Extraction (Stylistic Features)
!python dataset/feature_extraction.py

## 🚀 Training Models

Chúng ta sẽ train 2 mô hình để so sánh hiệu năng của Stylistic Features.

In [ ]:
# 🏃 7. Train BASELINE (PhoBERT only)
!python model/train_phobert.py --variant baseline --epochs 5 --batch-size 32

In [ ]:
# 🏃 8. Train PRO (PhoBERT + Stylistic Features)
!python model/train_phobert.py --variant features --epochs 5 --batch-size 32

## 📊 Final Results comparison

Sau khi train xong, bạn có thể xem kết quả tóm tắt Markdown trên console hoặc load history JSON từ `saved_models/artifacts/` để vẽ biểu đồ.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import glob

history_files = glob.glob("saved_models/artifacts/*_history.json")
all_results = []

for f in history_files:
    with open(f, 'r') as j:
        data = json.load(j)
        all_results.append({
            'Model': data['config']['variant'],
            'Best F1': data['best_f1'],
            'Best Acc': data['history'][data['best_epoch']-1]['val_accuracy'],
            'Best AUC': data['history'][data['best_epoch']-1]['val_auc']
        })

df_res = pd.DataFrame(all_results)
print("📊 Bảng so sánh kết quả:")
display(df_res)

df_res.set_index('Model')[['Best F1', 'Best Acc']].plot(kind='bar', figsize=(10, 5))
plt.title("Comparison: PhoBERT Baseline vs with Features")
plt.ylabel("Score")
plt.ylim(0, 1.0)
plt.show()